
# UVIF Educational Resilience Dynamics

## A Reproducible Framework for Equilibrium-Aware Learning Analytics

This notebook presents a reproducible implementation of a UVIF-based educational resilience framework for modeling learner-state evolution, temporal educational stability, and intervention-aware learning analytics.

The framework shifts educational AI from isolated prediction toward dynamic learner support by modeling:

- educational equilibrium,
- learner-state transitions,
- resilience and recovery,
- overload and instability,
- intervention effects,
- institutional support prioritization.

The notebook generates all tables, figures, and summary outputs required for manuscript-level analysis and repository documentation.



## Research Motivation

Conventional educational AI systems often focus on predicting outcomes such as failure risk, dropout probability, or performance level. While useful, prediction alone does not explain how learners move between stable, fragile, and critical educational states.

This notebook implements a UVIF-based framework that treats learning as a dynamic equilibrium process. Learner support is therefore modeled not only as a classification problem, but also as a stability, resilience, and intervention-governance problem.

The central question is:

> How can educational AI support learners by modeling equilibrium, instability, resilience, and recovery over time?



## UVIF Educational Equilibrium Formulation

The learner state is modeled through an educational equilibrium variable:

$$
E_{edu}(t+1)
=
E_{edu}(t)
+
\Delta_{learning}
+
\Delta_{engagement}
+
\Delta_{resilience}
-
\Delta_{overload}
-
\Delta_{instability}
+
\Delta_{intervention}
$$

where:

- $E_{edu}(t)$ is the learner equilibrium at time $t$,
- $\Delta_{learning}$ represents performance progression,
- $\Delta_{engagement}$ represents sustained participation,
- $\Delta_{resilience}$ represents recovery capacity,
- $\Delta_{overload}$ represents cognitive or workload stress,
- $\Delta_{instability}$ represents behavioral or performance volatility,
- $\Delta_{intervention}$ represents educational support.


In [ ]:

# ============================================================
# Cell 1 — Environment and Google Drive Initialization
# ============================================================

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    brier_score_loss,
    confusion_matrix,
    classification_report
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

if IN_COLAB:
    BASE_DIR = Path('/content/drive/MyDrive/Outputs/UVIF_Educational_Resilience_Dynamics')
else:
    BASE_DIR = Path.cwd() / 'Outputs' / 'UVIF_Educational_Resilience_Dynamics'

FIG_DIR = BASE_DIR / 'figures'
TABLE_DIR = BASE_DIR / 'tables'
OUTPUT_DIR = BASE_DIR / 'outputs'
DATA_DIR = BASE_DIR / 'data'

for d in [FIG_DIR, TABLE_DIR, OUTPUT_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def savefig(name):
    path = FIG_DIR / name
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'Saved figure: {path}')
    return path

print('Notebook initialized.')
print('Base directory:', BASE_DIR)



## Synthetic Longitudinal Learning Dataset

The notebook uses a reproducible synthetic longitudinal dataset. The purpose is not to claim empirical generalization from a specific institution, but to provide a controlled, transparent, and reusable framework for UVIF-based educational modeling.

Each learner is observed over multiple weeks using variables related to:

- performance,
- engagement,
- overload,
- instability,
- resilience,
- intervention support,
- UVIF equilibrium.


In [ ]:

# ============================================================
# Cell 2 — Generate Synthetic Longitudinal Educational Dataset
# ============================================================

n_learners = 1400
n_weeks = 16

records = []

for learner_id in range(n_learners):

    baseline_performance = np.clip(np.random.normal(0.65, 0.18), 0, 1)
    baseline_engagement = np.clip(np.random.normal(0.62, 0.20), 0, 1)
    baseline_resilience = np.clip(np.random.normal(0.60, 0.15), 0, 1)
    equilibrium = np.clip(np.random.normal(0.60, 0.15), 0, 1)

    for week in range(n_weeks):

        overload = np.clip(
            np.random.normal(0.35 + 0.20 * (1 - equilibrium), 0.12),
            0,
            1
        )

        instability = np.clip(
            np.random.normal(0.25 + 0.30 * overload, 0.10),
            0,
            1
        )

        intervention_support = np.clip(
            np.random.normal(0.30 + 0.40 * (equilibrium < 0.45), 0.10),
            0,
            1
        )

        engagement = np.clip(
            baseline_engagement
            - 0.25 * overload
            + 0.15 * intervention_support
            - 0.10 * instability
            + np.random.normal(0, 0.05),
            0,
            1
        )

        performance = np.clip(
            baseline_performance
            + 0.18 * engagement
            - 0.20 * overload
            - 0.10 * instability
            + np.random.normal(0, 0.05),
            0,
            1
        )

        resilience = np.clip(
            baseline_resilience
            + 0.15 * intervention_support
            - 0.20 * instability
            + np.random.normal(0, 0.05),
            0,
            1
        )

        equilibrium = np.clip(
            equilibrium
            + 0.10 * performance
            + 0.10 * engagement
            + 0.10 * resilience
            - 0.15 * overload
            - 0.15 * instability
            + 0.08 * intervention_support
            - 0.20,
            0,
            1
        )

        intervention_need = int(
            (equilibrium < 0.42)
            or (overload > 0.65)
            or (instability > 0.60)
        )

        records.append({
            'learner_id': learner_id,
            'week': week,
            'performance': performance,
            'engagement': engagement,
            'overload': overload,
            'instability': instability,
            'resilience': resilience,
            'intervention_support': intervention_support,
            'uvif_equilibrium': equilibrium,
            'intervention_need': intervention_need
        })

df = pd.DataFrame(records)

dataset_path = DATA_DIR / 'uvif_temporal_educational_dataset.csv'
df.to_csv(dataset_path, index=False)

print('Dataset shape:', df.shape)
print('Intervention-need rate:', round(df['intervention_need'].mean(), 4))
print('Saved dataset:', dataset_path)

df.head()



## Educational Equilibrium States

Learners are grouped into interpretable equilibrium states:

- **critical**: severe instability or low equilibrium,
- **fragile**: unstable learner state requiring close monitoring,
- **recovering**: positive but not fully stable trajectory,
- **stable**: favorable equilibrium and lower intervention need.

These states are intended to support educational interpretation and intervention planning.


In [ ]:

# ============================================================
# Cell 3 — Define Educational Equilibrium States
# ============================================================

def equilibrium_state(x):
    if x < 0.30:
        return 'critical'
    elif x < 0.50:
        return 'fragile'
    elif x < 0.75:
        return 'recovering'
    return 'stable'

state_order = ['critical', 'fragile', 'recovering', 'stable']

df['equilibrium_state'] = df['uvif_equilibrium'].apply(equilibrium_state)
df['equilibrium_state'] = pd.Categorical(
    df['equilibrium_state'],
    categories=state_order,
    ordered=True
)

state_summary = (
    df.groupby('equilibrium_state', observed=False)
      .agg(
          n=('learner_id', 'count'),
          intervention_need_rate=('intervention_need', 'mean'),
          mean_equilibrium=('uvif_equilibrium', 'mean'),
          mean_performance=('performance', 'mean'),
          mean_engagement=('engagement', 'mean'),
          mean_overload=('overload', 'mean'),
          mean_instability=('instability', 'mean'),
          mean_resilience=('resilience', 'mean')
      )
      .reset_index()
)

state_summary_path = TABLE_DIR / 'table_equilibrium_state_summary.csv'
state_summary.to_csv(state_summary_path, index=False)

print(state_summary.round(4))
print('Saved:', state_summary_path)



## Figure 1 — Educational Equilibrium Landscape

The landscape visualizes how engagement and overload jointly shape UVIF equilibrium. Regions of high engagement and lower overload are expected to form more favorable educational states, while high overload combined with weaker engagement indicates potential instability.


In [ ]:

# ============================================================
# Cell 4 — Educational Equilibrium Landscape
# ============================================================

plt.figure(figsize=(9, 6))

scatter = plt.scatter(
    df['engagement'],
    df['overload'],
    c=df['uvif_equilibrium'],
    alpha=0.35,
    s=18
)

plt.colorbar(scatter, label='UVIF equilibrium')
plt.xlabel('Engagement')
plt.ylabel('Overload')
plt.title('Educational Equilibrium Landscape')
plt.grid(alpha=0.25)

savefig('fig_educational_equilibrium_landscape.png')



## Learner-State Transition Matrix

The transition matrix estimates how learners move between equilibrium states from one week to the next. This table supports a dynamic interpretation of educational analytics by showing whether learners tend to remain stable, deteriorate, or recover.


In [ ]:

# ============================================================
# Cell 5 — Learner-State Transition Matrix
# ============================================================

transition_records = []

for learner_id, group in df.groupby('learner_id'):
    group = group.sort_values('week')
    states = group['equilibrium_state'].astype(str).values

    for i in range(len(states) - 1):
        transition_records.append({
            'from_state': states[i],
            'to_state': states[i + 1]
        })

transition_df = pd.DataFrame(transition_records)

transition_matrix = pd.crosstab(
    transition_df['from_state'],
    transition_df['to_state'],
    normalize='index'
).reindex(index=state_order, columns=state_order).fillna(0)

transition_matrix_path = TABLE_DIR / 'table_transition_matrix.csv'
transition_matrix.to_csv(transition_matrix_path)

print(transition_matrix.round(3))
print('Saved:', transition_matrix_path)



## Figure 2 — Learner-State Transition Heatmap

The heatmap provides a visual interpretation of state persistence, deterioration, and recovery. Diagonal dominance suggests persistence in a state, while off-diagonal movements reveal recovery or decline.


In [ ]:

# ============================================================
# Cell 6 — Transition Heatmap
# ============================================================

plt.figure(figsize=(7, 6))

plt.imshow(transition_matrix.values, aspect='auto')
plt.colorbar(label='Transition probability')
plt.xticks(range(len(state_order)), state_order, rotation=30)
plt.yticks(range(len(state_order)), state_order)
plt.xlabel('Next state')
plt.ylabel('Current state')
plt.title('Learner-State Transition Matrix')

for i in range(len(state_order)):
    for j in range(len(state_order)):
        plt.text(j, i, f'{transition_matrix.values[i, j]:.2f}',
                 ha='center', va='center')

savefig('fig_transition_matrix_heatmap.png')



## Temporal Stability and Resilience Analysis

For each learner, the notebook computes:

- mean equilibrium,
- equilibrium volatility,
- resilience level,
- overload exposure,
- instability level,
- stability index.

The stability index is defined as:

$$
S_i = \bar{E}_{edu,i} - \sigma(E_{edu,i})
$$

where higher values indicate stronger and more stable educational equilibrium.


In [ ]:

# ============================================================
# Cell 7 — Temporal Stability and Resilience Analysis
# ============================================================

learner_stability = (
    df.groupby('learner_id')
      .agg(
          equilibrium_mean=('uvif_equilibrium', 'mean'),
          equilibrium_std=('uvif_equilibrium', 'std'),
          resilience_mean=('resilience', 'mean'),
          overload_mean=('overload', 'mean'),
          instability_mean=('instability', 'mean'),
          intervention_need_rate=('intervention_need', 'mean')
      )
      .reset_index()
)

learner_stability['stability_index'] = (
    learner_stability['equilibrium_mean']
    - learner_stability['equilibrium_std']
)

learner_stability['resilience_group'] = pd.qcut(
    learner_stability['resilience_mean'],
    q=3,
    labels=['low_resilience', 'moderate_resilience', 'high_resilience']
)

stability_summary = (
    learner_stability.groupby('resilience_group', observed=False)
    .agg(
        n=('learner_id', 'count'),
        mean_equilibrium=('equilibrium_mean', 'mean'),
        mean_equilibrium_volatility=('equilibrium_std', 'mean'),
        mean_stability_index=('stability_index', 'mean'),
        mean_overload=('overload_mean', 'mean'),
        mean_instability=('instability_mean', 'mean'),
        mean_intervention_need_rate=('intervention_need_rate', 'mean')
    )
    .reset_index()
)

learner_stability_path = TABLE_DIR / 'table_learner_temporal_stability.csv'
stability_summary_path = TABLE_DIR / 'table_resilience_group_summary.csv'

learner_stability.to_csv(learner_stability_path, index=False)
stability_summary.to_csv(stability_summary_path, index=False)

print(stability_summary.round(4))
print('Saved:', learner_stability_path)
print('Saved:', stability_summary_path)



## Figure 3 — Temporal Educational Equilibrium Dynamics

This figure summarizes how average educational equilibrium evolves over time. It helps assess whether the simulated educational system moves toward stability, deterioration, or sustained fragility.


In [ ]:

# ============================================================
# Cell 8 — Temporal Educational Equilibrium Dynamics
# ============================================================

weekly_summary = (
    df.groupby('week')
      .agg(
          mean_equilibrium=('uvif_equilibrium', 'mean'),
          std_equilibrium=('uvif_equilibrium', 'std'),
          mean_overload=('overload', 'mean'),
          mean_resilience=('resilience', 'mean'),
          intervention_need_rate=('intervention_need', 'mean')
      )
      .reset_index()
)

weekly_summary_path = TABLE_DIR / 'table_weekly_equilibrium_summary.csv'
weekly_summary.to_csv(weekly_summary_path, index=False)

plt.figure(figsize=(10, 5))

plt.plot(
    weekly_summary['week'],
    weekly_summary['mean_equilibrium'],
    linewidth=3,
    label='Mean UVIF equilibrium'
)

plt.fill_between(
    weekly_summary['week'],
    weekly_summary['mean_equilibrium'] - weekly_summary['std_equilibrium'],
    weekly_summary['mean_equilibrium'] + weekly_summary['std_equilibrium'],
    alpha=0.2,
    label='±1 standard deviation'
)

plt.xlabel('Week')
plt.ylabel('UVIF equilibrium')
plt.title('Temporal Educational Equilibrium Dynamics')
plt.legend()
plt.grid(alpha=0.25)

savefig('fig_temporal_equilibrium_dynamics.png')

print('Saved:', weekly_summary_path)



## Figure 4 — Resilience and Stability Relationship

The relationship between resilience and stability index helps identify whether learners with stronger resilience also maintain more stable educational equilibrium.


In [ ]:

# ============================================================
# Cell 9 — Resilience and Stability Relationship
# ============================================================

plt.figure(figsize=(8, 6))

plt.scatter(
    learner_stability['resilience_mean'],
    learner_stability['stability_index'],
    alpha=0.45,
    s=20
)

plt.xlabel('Mean resilience')
plt.ylabel('Stability index')
plt.title('Relationship Between Resilience and Educational Stability')
plt.grid(alpha=0.25)

savefig('fig_resilience_stability_relationship.png')



## Intervention Governance Priorities

The framework converts learner dynamics into support priorities. These priorities are not intended to replace educators; they are designed to support transparent decision-making and resource allocation.


In [ ]:

# ============================================================
# Cell 10 — Intervention Governance Priorities
# ============================================================

governance = learner_stability.copy()

conditions = [
    governance['equilibrium_mean'] < 0.35,
    governance['equilibrium_mean'] < 0.50,
    governance['equilibrium_mean'] < 0.70
]

choices = [
    'urgent_support',
    'high_priority',
    'monitoring'
]

governance['governance_priority'] = np.select(
    conditions,
    choices,
    default='stable_monitoring'
)

governance_summary = (
    governance.groupby('governance_priority')
    .agg(
        n=('learner_id', 'count'),
        mean_equilibrium=('equilibrium_mean', 'mean'),
        mean_stability_index=('stability_index', 'mean'),
        mean_overload=('overload_mean', 'mean'),
        mean_instability=('instability_mean', 'mean'),
        mean_intervention_need_rate=('intervention_need_rate', 'mean')
    )
    .reset_index()
)

governance_path = TABLE_DIR / 'table_governance_priorities.csv'
governance_summary_path = TABLE_DIR / 'table_governance_priority_summary.csv'

governance.to_csv(governance_path, index=False)
governance_summary.to_csv(governance_summary_path, index=False)

print(governance_summary.round(4))
print('Saved:', governance_path)
print('Saved:', governance_summary_path)



## Counterfactual Educational Support Simulation

The notebook evaluates four transparent counterfactual support scenarios:

1. engagement support,
2. workload reduction,
3. resilience reinforcement,
4. combined support.

Each scenario estimates the expected change in educational equilibrium.


In [ ]:

# ============================================================
# Cell 11 — Counterfactual Educational Support Scenarios
# ============================================================

def apply_counterfactual(data, scenario):
    temp = data.copy()

    if scenario == 'engagement_support':
        temp['engagement_cf'] = np.clip(temp['engagement'] + 0.10, 0, 1)
        temp['overload_cf'] = temp['overload']
        temp['resilience_cf'] = temp['resilience']
        temp['intervention_support_cf'] = temp['intervention_support'] + 0.05

    elif scenario == 'workload_reduction':
        temp['engagement_cf'] = temp['engagement']
        temp['overload_cf'] = np.clip(temp['overload'] - 0.12, 0, 1)
        temp['resilience_cf'] = temp['resilience']
        temp['intervention_support_cf'] = temp['intervention_support'] + 0.05

    elif scenario == 'resilience_reinforcement':
        temp['engagement_cf'] = temp['engagement']
        temp['overload_cf'] = temp['overload']
        temp['resilience_cf'] = np.clip(temp['resilience'] + 0.12, 0, 1)
        temp['intervention_support_cf'] = temp['intervention_support'] + 0.10

    elif scenario == 'combined_support':
        temp['engagement_cf'] = np.clip(temp['engagement'] + 0.10, 0, 1)
        temp['overload_cf'] = np.clip(temp['overload'] - 0.12, 0, 1)
        temp['resilience_cf'] = np.clip(temp['resilience'] + 0.12, 0, 1)
        temp['intervention_support_cf'] = np.clip(temp['intervention_support'] + 0.15, 0, 1)

    temp['uvif_equilibrium_cf'] = np.clip(
        temp['uvif_equilibrium']
        + 0.10 * (temp['engagement_cf'] - temp['engagement'])
        - 0.10 * (temp['overload_cf'] - temp['overload'])
        + 0.10 * (temp['resilience_cf'] - temp['resilience'])
        + 0.08 * (temp['intervention_support_cf'] - temp['intervention_support']),
        0,
        1
    )

    return temp

scenario_rows = []

for scenario in [
    'engagement_support',
    'workload_reduction',
    'resilience_reinforcement',
    'combined_support'
]:
    temp = apply_counterfactual(df, scenario)

    scenario_rows.append({
        'scenario': scenario,
        'mean_equilibrium_before': df['uvif_equilibrium'].mean(),
        'mean_equilibrium_after': temp['uvif_equilibrium_cf'].mean(),
        'delta_equilibrium': temp['uvif_equilibrium_cf'].mean() - df['uvif_equilibrium'].mean(),
        'critical_rate_before': (df['equilibrium_state'].astype(str) == 'critical').mean(),
        'critical_rate_after': (temp['uvif_equilibrium_cf'] < 0.30).mean()
    })

counterfactual_summary = pd.DataFrame(scenario_rows)

counterfactual_path = TABLE_DIR / 'table_counterfactual_support_scenarios.csv'
counterfactual_summary.to_csv(counterfactual_path, index=False)

print(counterfactual_summary.round(4))
print('Saved:', counterfactual_path)



## Figure 5 — Counterfactual Equilibrium Gains

This figure compares the estimated equilibrium gain associated with different support strategies.


In [ ]:

# ============================================================
# Cell 12 — Counterfactual Equilibrium Gain Figure
# ============================================================

plt.figure(figsize=(9, 5))

plt.bar(
    counterfactual_summary['scenario'],
    counterfactual_summary['delta_equilibrium']
)

plt.ylabel('Mean equilibrium gain')
plt.xlabel('Counterfactual support scenario')
plt.title('Counterfactual Educational Support Effects')
plt.xticks(rotation=25, ha='right')
plt.grid(axis='y', alpha=0.25)

savefig('fig_counterfactual_support_effects.png')



## Predictive Decision Layer

A lightweight predictive model is included to estimate intervention need. The purpose of this model is not to dominate benchmarks, but to connect equilibrium-based learner modeling with decision-oriented educational analytics.


In [ ]:

# ============================================================
# Cell 13 — Predictive Educational Decision Layer
# ============================================================

features = [
    'performance',
    'engagement',
    'overload',
    'instability',
    'resilience',
    'intervention_support',
    'uvif_equilibrium'
]

X = df[features]
y = df['intervention_need']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)

model = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', GradientBoostingClassifier(random_state=RANDOM_STATE))
])

model.fit(X_train, y_train)

pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

predictive_results = pd.DataFrame({
    'model': ['GradientBoosting_UVIF_Decision_Layer'],
    'accuracy': [accuracy_score(y_test, pred)],
    'balanced_accuracy': [balanced_accuracy_score(y_test, pred)],
    'f1': [f1_score(y_test, pred)],
    'roc_auc': [roc_auc_score(y_test, proba)],
    'brier_score': [brier_score_loss(y_test, proba)]
})

predictive_path = TABLE_DIR / 'table_predictive_decision_results.csv'
predictive_results.to_csv(predictive_path, index=False)

print(predictive_results.round(4))
print('Saved:', predictive_path)

print('\nClassification report:')
print(classification_report(y_test, pred))



## Representative Learner Trajectories

The following figure illustrates how individual learners may follow different equilibrium trajectories. This supports longitudinal interpretation beyond single-point prediction.


In [ ]:

# ============================================================
# Cell 14 — Representative Learner Equilibrium Trajectories
# ============================================================

selected = np.random.choice(df['learner_id'].unique(), 8, replace=False)

plt.figure(figsize=(10, 6))

for learner_id in selected:
    temp = df[df['learner_id'] == learner_id].sort_values('week')

    plt.plot(
        temp['week'],
        temp['uvif_equilibrium'],
        alpha=0.85,
        linewidth=2
    )

plt.xlabel('Week')
plt.ylabel('UVIF equilibrium')
plt.title('Representative Learner Equilibrium Trajectories')
plt.grid(alpha=0.25)

savefig('fig_representative_equilibrium_trajectories.png')



## Manuscript-Oriented Output Summary

The final cell writes a concise summary of generated tables, figures, and key framework contributions.


In [ ]:

# ============================================================
# Cell 15 — Save Outputs Summary
# ============================================================

summary = f'''
UVIF Educational Resilience Dynamics — Reproducible Outputs Summary

Base directory:
{BASE_DIR}

Dataset:
- data/uvif_temporal_educational_dataset.csv

Main scientific contributions:
1. UVIF-based educational equilibrium modeling.
2. Longitudinal learner-state transition analysis.
3. Temporal stability and resilience analysis.
4. Intervention governance prioritization.
5. Counterfactual educational support simulation.
6. Predictive decision layer for intervention need.
7. Reproducible manuscript-ready figures and tables.

Main tables:
- table_equilibrium_state_summary.csv
- table_transition_matrix.csv
- table_learner_temporal_stability.csv
- table_resilience_group_summary.csv
- table_weekly_equilibrium_summary.csv
- table_governance_priorities.csv
- table_governance_priority_summary.csv
- table_counterfactual_support_scenarios.csv
- table_predictive_decision_results.csv

Main figures:
- fig_educational_equilibrium_landscape.png
- fig_transition_matrix_heatmap.png
- fig_temporal_equilibrium_dynamics.png
- fig_resilience_stability_relationship.png
- fig_counterfactual_support_effects.png
- fig_representative_equilibrium_trajectories.png

Interpretive summary:
The notebook reframes educational AI as a dynamic equilibrium and resilience modeling problem.
Rather than treating intervention as a static prediction output, the framework models learner-state evolution, stability, recovery, and support prioritization over time.
'''

summary_path = OUTPUT_DIR / 'outputs_summary.txt'

with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(summary)

print(summary)
print('Saved:', summary_path)
